# Hybrid QLSTM Model Design

## 1. Design Overview
We will integrate a Variational Quantum Circuit (VQC) as an initial feature extraction layer before feeding the data into a Classical LSTM network.

### Architecture:
1. **Input**: Sequence of 5 features (Open, High, Low, Close, Volume).
2. **Quantum Layer (VQC)**: Processes input features using 4 qubits.
3. **LSTM Layers**: Two stacked LSTM layers (50 units each) to capture temporal dependencies.
4. **Output**: Dense layer (Sigmoid) for binary classification (Up/Down).

## 2. Import Libraries

In [ ]:
import pennylane as qml
from pennylane import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input, TimeDistributed, Layer

# Note: Standard PennyLane KerasLayer might fail with Keras 3 (TF 2.16+).
# We will define a custom layer to ensure compatibility.

## 3. Define Quantum Circuit

In [ ]:
n_qubits = 5  # Matching feature size roughly
n_layers = 2
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(w)) for w in range(n_qubits)]

## 4. Define Custom Quantum Layer
This wrapper ensures we can use the QNode within Keras even if libraries mismatch.

In [ ]:
class QuantumLayer(Layer):
    def __init__(self, qnode, weight_shapes, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.qnode = qnode
        self.output_dim = output_dim
        self.w = self.add_weight(
            shape=weight_shapes["weights"],
            initializer="random_normal",
            trainable=True,
            name="weights"
        )

    def call(self, inputs):
        return tf.convert_to_tensor(self.qnode(inputs, self.w))

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.output_dim)

## 5. Build Hybrid Model

In [ ]:
def create_hybrid_model(input_shape):
    weight_shapes = {"weights": (n_layers, n_qubits)}
    qlayer = QuantumLayer(quantum_circuit, weight_shapes, output_dim=n_qubits)
    
    model = Sequential()
    # TimeDistributed applies VQC to each time step independently
    model.add(TimeDistributed(qlayer, input_shape=input_shape))
    model.add(LSTM(50, return_sequences=True))
    model.add(LSTM(50))
    model.add(Dense(1, activation='sigmoid'))
    return model

model = create_hybrid_model(input_shape=(60, n_qubits))
model.summary()
print("Hybrid Model Design Defined.")